In [0]:
%python
print(spark.version)

3.5.2


In [0]:
%python
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, concat, lit, initcap, hour, round, floor, unix_timestamp, dayofweek, count, avg, max, min, sum
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType,TimestampType

#schema for yellow files
yellow_taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])
#schema for green files
green_taxi_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("lpep_pickup_datetime", TimestampType(), True),
    StructField("lpep_dropoff_datetime", TimestampType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),	
    StructField("passenger_count", LongType(), True),	
    StructField("trip_distance", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("ehail_fee", IntegerType(), True),	
    StructField("improvement_surcharge", DoubleType(), True),	
	StructField("total_amount", DoubleType(), True),
    StructField("payment_type", DoubleType(), True),
    StructField("trip_type", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
])

# --- 1. Load Data ---
S3_BUCKET_NAME = "robot-dreams-source-data"
YELLOW_TAXI_PATH = f"s3a://{S3_BUCKET_NAME}/home-work-1-unified/nyc_taxi/yellow/"
GREEN_TAXI_PATH = f"s3a://{S3_BUCKET_NAME}/home-work-1-unified/nyc_taxi/green/"

#read yellow
#filter on read stage = small but optimization
print(f"Loading Yellow Taxi data from: {YELLOW_TAXI_PATH}")
yellow_taxi_df = spark.read \
                     .option("mergeSchema", "true") \
                     .option("recursiveFileLookup", "true") \
                     .schema(yellow_taxi_schema).parquet(YELLOW_TAXI_PATH) \
                     .withColumn("taxi_type", lit('yellow')) \
                     .filter(
                        (col("trip_distance") > 0.1) &
                        (col("fare_amount") > 2)
                    )

#yellow_unified = yellow_taxi_df.select(col("VendorID").cast("long"))

#read green
#filter on read stage = small but optimization
#filter anomalies
print(f"Loading Green Taxi data from: {GREEN_TAXI_PATH}")
green_taxi_df = spark.read \
                     .option("mergeSchema", "true") \
                     .option("recursiveFileLookup", "true") \
                     .schema(green_taxi_schema).parquet(GREEN_TAXI_PATH) \
                     .withColumn("taxi_type", lit('green')) \
                     .filter(
                        (col("trip_distance") > 0.1) &
                        (col("fare_amount") > 2)
                    )
#green_unified = green_taxi_df.select(col("VendorID").cast("long"))

print(f"Yellow Taxi data loaded. Number of records: {yellow_taxi_df.count()}")
print(f"Green Taxi data loaded. Number of records: {green_taxi_df.count()}")
print("Schema:")
green_taxi_df.printSchema()

#unoin dataframes green + yellow
raw_trips_df = yellow_taxi_df.unionByName(green_taxi_df, allowMissingColumns=True)

#raw_trips_df.write.format("delta") \
#    .mode("overwrite") \
#    .saveAsTable("vyaskov_catalog.trips_schema.raw_trips")

Loading Yellow Taxi data from: s3a://robot-dreams-source-data/home-work-1-unified/nyc_taxi/yellow/
Loading Green Taxi data from: s3a://robot-dreams-source-data/home-work-1-unified/nyc_taxi/green/
Yellow Taxi data loaded. Number of records: 736860719
Green Taxi data loaded. Number of records: 75994464
Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: integer (nullable = true)
 |-- improvement_surch

In [0]:
%python
#add new columns: extract new info from existing columns
#duration for another anomalies filter

add_duration = raw_trips_df.withColumn("duration_min", (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60)\
    .withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day_of_week", dayofweek("tpep_pickup_datetime"))

#another anomalies filter
filter_duration = add_duration.filter("duration_min > 1")

print(f"After UNION and filters: {filter_duration.count()}")

After UNION and filters: 734844553


In [0]:
%python

lookup_path = f"s3://robot-dreams-source-data/home-work-1-unified/nyc_taxi/taxi_zone_lookup.csv"

#read csv, no comments
zones_df = spark.read.option("header", True).csv(lookup_path) \
    .select(
        col("LocationID").cast("int").alias("location_id"),
        col("Zone").alias("zone")
    )

#we need to join twice, so create two dfs to awoid name conflict on join stage
zones_pickup = zones_df.alias("pickup_zones")
zones_dropoff = zones_df.alias("dropoff_zones")

#join dictionary and add informative columns
result_df = (
    filter_duration
    .join(zones_pickup, col("PULocationID") == col("pickup_zones.location_id"), "left")
    .join(zones_dropoff, col("DOLocationID") == col("dropoff_zones.location_id"), "left")
    .withColumn("pickup_zone", when(col("pickup_zones.zone").isNull(), "Unknown").otherwise(col("pickup_zones.zone")))
    .withColumn("dropoff_zone", when(col("dropoff_zones.zone").isNull(), "Unknown").otherwise(col("dropoff_zones.zone")))
    .select(
        # залишаємо потрібні колонки з filter_duration
        *[c for c in filter_duration.columns],
        col("pickup_zone"),
        col("dropoff_zone")
    )
)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS vyaskov_catalog
MANAGED LOCATION 's3://databricks-vyaskov/';

In [0]:
%sql
USE CATALOG vyaskov_catalog;
SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
vyaskov_catalog,default


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS vyaskov_catalog.trips_schema;

In [0]:
%python
result_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("vyaskov_catalog.trips_schema.raw_trips")

In [0]:
%python
#use aggregation and conditional expression "case when..."
zone_summary = result_df.groupBy("pickup_zone").agg(
    count("*").alias("total_trips"),
    round(avg("trip_distance"), 2).alias("avg_trip_distance"),
    round(avg("total_amount"), 2).alias("avg_total_amount"),
    round(avg("tip_amount"), 2).alias("avg_tip_amount"),
    max("trip_distance").alias("max_trip_distance"),
    min("tip_amount").alias("min_tip_amount"),
    sum(when(col("taxi_type") == "yellow", 1).otherwise(0)).alias("yellow_count"),
    sum(when(col("taxi_type") == "green", 1).otherwise(0)).alias("green_count")
)




#create delta table zone_summary
zone_summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("vyaskov_catalog.trips_schema.zone_summary")

In [0]:
zone_day_stats = result_df.groupBy("pickup_zone", "pickup_day_of_week").agg(
    count("*").alias("total_trips"),
    round(sum(when(col("total_amount") > 30, 1).otherwise(0)) / count("*"), 4).alias("high_fare_share")
)

In [0]:
%python
#create delta table zone_summary
zone_day_stats.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("vyaskov_catalog.trips_schema.zone_days_summary")